### 1. Load configuration
Load catalog names, schemas, and the source table list. Create target schemas if they do not exist.

In [0]:
%run ./00_config

### 2. Load Bronze tables
Copy all eight TPC-H tables into Bronze Delta tables without changing source values. Add ingestion metadata and validate row counts. Overwrite existing tables on each run.

In [0]:
from pyspark.sql import functions as F
from uuid import uuid4

run_id = str(uuid4())
results = []

for table in TABLES:
    source = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}"
    target = f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{table}"

    source_df = spark.table(source)

    bronze_df = (
        source_df
        .withColumn("_loaded_at", F.current_timestamp())
        .withColumn("_source_table", F.lit(source))
        .withColumn("_run_id", F.lit(run_id))
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target)
    )

    source_count = source_df.count()
    bronze_count = spark.table(target).count()

    results.append((
        table,
        source_count,
        bronze_count,
        source_count == bronze_count,
    ))

    print(f"{table}: source={source_count}, bronze={bronze_count}")

display(spark.createDataFrame(
    results,
    "table_name string, source_rows long, bronze_rows long, counts_match boolean",
))

assert all(row[3] for row in results), "Кількість рядків не збігається"

### 3. Preview Bronze data
Display ten rows from the Bronze part table for a quick visual inspection.

In [0]:
display(
    spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.part").limit(10)
)

### 4. Validate monetary totals
Compare total order prices, line item amounts, and revenue after discounts between source and Bronze. Fail the run if any total differs.

In [0]:
from pyspark.sql import functions as F

metrics = [
    ("orders", "total_order_price", "o_totalprice"),
    ("lineitem", "total_extended_price", "l_extendedprice"),
    (
        "lineitem",
        "net_revenue",
        "l_extendedprice * (1 - l_discount)",
    ),
]

money_checks = []

for table, metric, expression in metrics:
    source = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}"
    target = f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{table}"

    def total(table_name):
        return (
            spark.table(table_name)
            .agg(
                F.sum(
                    F.expr(expression).cast("decimal(28, 6)")
                ).alias("total")
            )
            .first()["total"]
        )

    source_total = total(source)
    bronze_total = total(target)

    assert source_total is not None and bronze_total is not None, (
        f"Missing total for {metric}"
    )

    difference = bronze_total - source_total

    money_checks.append((
        metric,
        str(source_total),
        str(bronze_total),
        str(difference),
        difference == 0,
    ))

display(
    spark.createDataFrame(
        money_checks,
        """
        metric string,
        source_total string,
        bronze_total string,
        difference string,
        totals_match boolean
        """,
    )
)

assert all(row[4] for row in money_checks), (
    "Source and Bronze monetary totals do not match"
)